"""

Created on Feb 28 2026

by Minde An
mindean@mit.edu


"""

In [ ]:
## In[0]:
# Import necessary packages
import pandas as pd
import numpy as np
from scipy.integrate import solve_ivp
import matplotlib.pyplot as plt
from pathlib import Path
from mit_inversions.inversion.inversion import analytical_inversion


In [ ]:
# In[1]:
# Read data at Cape Grim and Trinidad head

# Cape Grim, 41S
df_CGO = pd.read_csv(f'{Path.cwd()}/AGAGE-GCMD_CGO_cfc-11_mon.txt',  sep=r"\s+", comment='#', names=['t', 'year','month', 'cfc11', 'uncertainty','numb'])
df_CGO['date'] = pd.to_datetime(df_CGO[['year', 'month']].assign(day=1))
df_CGO.set_index('date', inplace=True)
df_CGO = df_CGO.drop(columns=['t', 'year', 'month', 'numb'])
df_CGO=df_CGO.resample('YE').mean()


# Trinidad head, 41N
df_THD = pd.read_csv(f'{Path.cwd()}/AGAGE-GCMD_THD_cfc-11_mon.txt',  sep=r"\s+", comment='#', names=['t', 'year','month', 'cfc11', 'uncertainty','numb'])
df_THD['date'] = pd.to_datetime(df_THD[['year', 'month']].assign(day=1))
df_THD.set_index('date', inplace=True)
df_THD = df_THD.drop(columns=['t', 'year', 'month', 'numb'])
df_THD=df_THD.resample('YE').mean()

df_merge = pd.merge(df_CGO, df_THD, on='date', suffixes=('_CGO', '_THD'), how='inner')
df_merge['differences'] = df_merge['cfc11_THD'] - df_merge['cfc11_CGO']
df_merge.index = pd.to_datetime(df_merge.index.year.astype(str) + '-01-01')

del df_CGO, df_THD

In [ ]:
# In[2]:

# Define 3-box model
def boxModel(t,x, E,T_st, T_ts, T_ns, T_L):
    '''
    E: emission rate in each box
    T_st: stratosphere residence time
    T_ts: troposphere residence time
    T_ns: North-South hemisphere exchange time
    T_L: lifetime related to the loss rate in each box
    
    '''
    M = 5.15e21/28.9 #total moles of air
    m = np.array([0.15,0.425,0.425])*M #air moles in each box
    L = np.nan_to_num(1/T_L, nan=0.0) #calculate the loss rate in each box
    
    # mass balance equations
    dxdt=[0,0,0] # covert to ppt
    dxdt[0] = 1/m[0] * ( E[0][int(t)]*1e12  - m[0]*x[0]*L[0] + (m[1]*x[1]+ m[2]*x[2])/T_ts - m[0]*x[0]/T_st )
    dxdt[1] = 1/m[1] * ( E[1][int(t)]*1e12  - m[1]*x[1]*L[1] + m[0]*x[0]*0.5 /T_st - m[1]*x[1]/T_ts + (m[2]*x[2] - m[1]*x[1])/T_ns )
    dxdt[2] = 1/m[2] * ( E[2][int(t)]*1e12  - m[2]*x[2]*L[2] + m[0]*x[0]*0.5 /T_st - m[2]*x[2]/T_ts + (m[1]*x[1] - m[2]*x[2])/T_ns )
    return dxdt

In [ ]:
# In[3]:

## Matrix version 

# estimate emissions for 2008-2022
n_years = 14

# prior emissions [Gg/yr]
N_prior=45 # Gg/yr
S_prior=5
x_error=0.5

xa=np.tile([20,5],n_years).reshape(2*n_years,1) # prior emissions matrix : changes on top of prior
# prior covariance
P = np.zeros((2*n_years,2*n_years))
np.fill_diagonal(P,np.tile([(20*x_error)**2,(5*x_error)**2],n_years))


# Get observation

y = df_merge.loc["2008":"2021",['cfc11_THD','cfc11_CGO']].values.reshape(2*n_years,1)
# obs covariance
R = np.zeros((2*n_years,2*n_years))
np.fill_diagonal(R,(df_merge.loc["2008":"2021",['uncertainty_THD','uncertainty_CGO']].values**2).reshape(2*n_years,1) )

### Calculate Jacobian matrix using 3-box model (NH-SH to NH-emis)

x0=np.array([204, 245, 243])
T_st=1.5
T_ts=8.6
T_ns=1.1
T_L=np.array([7.8, np.nan, np.nan])
tspan = np.array([0,14])
t_eval = np.arange(0,15)
E=np.array([np.zeros_like(t_eval), [N_prior*1e9/137]*(n_years+1), [S_prior*1e9/137]*(n_years+1)])
sol_ref = solve_ivp(boxModel, tspan, x0, t_eval=t_eval, method='LSODA', args=(E,T_st, T_ts, T_ns, T_L)   )
H0 = np.zeros((2*n_years,2*n_years))

for sens_y in np.arange(0,n_years):
    # for NH
    E_pert_N = E.copy()
    E_pert_N[1][sens_y] *= 1.1
    sol_pert_N = solve_ivp(boxModel, tspan, x0, t_eval=t_eval, method='LSODA', args=(E_pert_N,T_st, T_ts, T_ns, T_L)   )
    H0[2*sens_y:,2*sens_y] = (np.column_stack((sol_pert_N.y[1,1:]-sol_ref.y[1,1:],sol_pert_N.y[2,1:]-sol_ref.y[2,1:])).flatten()/(N_prior*0.1))[2*sens_y:]

    # for SH
    E_pert_S = E.copy()
    E_pert_S[2][sens_y] *= 1.1
    sol_pert_S = solve_ivp(boxModel, tspan, x0, t_eval=t_eval, method='LSODA', args=(E_pert_S,T_st, T_ts, T_ns, T_L)   )
    H0[2*sens_y:,2*sens_y+1] = (np.column_stack((sol_pert_S.y[1,1:]-sol_ref.y[1,1:],sol_pert_S.y[2,1:]-sol_ref.y[2,1:])).flatten()/(S_prior*0.1))[2*sens_y:]



In [ ]:
y_i = y - sol_ref.y[1:3,1:].flatten('F').reshape(2*n_years,1)

In [ ]:
H0.shape, y_i.shape, R.shape, P.shape, xa.shape

In [ ]:
xhat, ak, shat=analytical_inversion(H0, y_i, R, xa, P)

In [ ]:
xhat_S = xhat[1::2] + S_prior
xhat_N = xhat[0::2] + N_prior
shat_S = np.diagonal(shat)[1::2]
shat_N = np.diagonal(shat)[0::2]
ak_S = np.diagonal(ak)[1::2]
ak_N = np.diagonal(ak)[0::2]

In [ ]:
# In[4]:
# Plot the results for Matrix version

year_range = pd.date_range(start='2008-01-01', periods=12, freq='YS')

fig = plt.figure()
plt.ylabel('CFC-11 Emissions (Gg/yr)')
plt.xlabel('Year')

# Plot prior and posterior emissions

plt.plot(year_range,[N_prior] * len(year_range),'b-', label='N_prior'  )
plt.fill_between(year_range, [N_prior-10]* len(year_range), [N_prior+10]* len(year_range), alpha=0.3)
plt.plot(year_range,[S_prior] * len(year_range),'g-', label='S_prior'  )
plt.fill_between(year_range, [S_prior-2.5]* len(year_range), [S_prior+2.5]* len(year_range), alpha=0.3)


plt.errorbar(year_range,xhat_N.flatten()[:-2],yerr=np.sqrt(shat_N).flatten()[:-2], marker='o',color='b',ecolor='b',label='posteror'  )
plt.errorbar(year_range,xhat_S.flatten()[:-2],yerr=np.sqrt(shat_S).flatten()[:-2], marker='o',color='g',ecolor='g',label='posteror'  )


plt.ylim([0,72])
plt.legend()
plt.show()




In [ ]:
# In[5]:
# Plot the error reduction for Matrix version

year_range = pd.date_range(start='2008-01-01', periods=14, freq='YS')

fig = plt.figure()
plt.ylabel('Error reduction')
plt.xlabel('Year')
plt.plot(year_range,ak_S,'b--', label='error_reduction_SH'  )
plt.plot(year_range,ak_N,'g--', label='error_reduction_NH'  )
plt.ylim([0,1])

plt.show()

In [ ]:
# In[6]:
# Plot the mole fractions for Matrix inversion

# Do farward run
t_year = np.arange(0,14) ## run for 2008-2022
x0=np.array([204, 245, 243])
T_st=1.5
T_ts=8.6
T_ns=1.1
T_L=np.array([7.8, np.nan, np.nan])
E_prior=np.array([[S_prior*1e9/137]*len(t_year), [N_prior*1e9/137]*len(t_year), np.zeros_like(t_year)])
E_posterior=np.array([xhat_S.flatten()*1e9/137, xhat_N.flatten()*1e9/137, np.zeros_like(t_year)])

prior_simulate = solve_ivp(boxModel, [0,13], x0, t_eval=t_year, method='LSODA', args=(E_prior,T_st, T_ts, T_ns, T_L)   )
NH_prior = prior_simulate.y[1,:]
SH_prior = prior_simulate.y[2,:]
posterior_simulate = solve_ivp(boxModel, [0,13], x0, t_eval=t_year, method='LSODA', args=(E_posterior,T_st, T_ts, T_ns, T_L)   )
NH_posterior = posterior_simulate.y[1,:]
SH_posterior = posterior_simulate.y[2,:]

fig = plt.figure()
year_range = pd.date_range(start='2008-01-01', periods=14, freq='YS')



plt.plot(df_merge.index,df_merge['cfc11_CGO'],'k-', label='Cape Grim'  )
plt.fill_between(df_merge.index, df_merge['cfc11_CGO']-df_merge['uncertainty_CGO'], df_merge['cfc11_CGO']+df_merge['uncertainty_CGO'], alpha=0.5)
plt.plot(df_merge.index,df_merge['cfc11_THD'], 'g-', label='Trinidad Head')
plt.fill_between(df_merge.index, df_merge['cfc11_THD']-df_merge['uncertainty_THD'], df_merge['cfc11_THD']+df_merge['uncertainty_THD'], alpha=0.5)

plt.plot(year_range, NH_prior , 'g--', label='NH prior')
plt.plot(year_range, SH_prior , 'k--', label='SH prior')
plt.plot(year_range, NH_posterior, 'y--', label='NH posterior') 
plt.plot(year_range, SH_posterior, 'r--', label='SH posterior')
plt.xlim([pd.Timestamp('2007-01-01'), pd.Timestamp('2021-12-31')])
plt.ylim([210, 250])
plt.legend()

plt.ylabel('CFC-11 [ppt]')
plt.xlabel('Year')
plt.show()



In [ ]:
# In[7]:
# Plot the gradient for Matrix version
# 
fig = plt.figure()
plt.ylabel('CFC-11 NH-SH gradients [ppt]')
plt.xlabel('Year')

plt.plot(df_merge.index, df_merge['differences'], 'k-', label='THD-CGO')
plt.plot(year_range, NH_prior-SH_prior, 'g-', label='prior NH-SH')
plt.plot(year_range, NH_posterior-SH_posterior, 'r-', label='posterior NH-SH')
plt.xlim([pd.Timestamp('2007-01-01'), pd.Timestamp('2021-12-31')])
plt.ylim([0, 6])
plt.legend()
plt.show()

